In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"
GOLD_PATH = "abfss://gold@pravdatalake.dfs.core.windows.net"
GOLD_TABLE_PATH = f"{GOLD_PATH}/dim_automaker"
GOLD_TABLE_NAME = "vehicle_sales.gold.dim_automaker"

In [0]:
silver_basic = spark.read.format("delta").load(f"{SILVER_PATH}/basic_table")

In [0]:
dim_automaker_updates = (
    silver_basic
    .select("Automaker_ID", "Automaker")
    .dropDuplicates(["Automaker_ID"])
    .withColumn("gold_updated_timestamp", current_timestamp())
)

In [0]:
dim_automaker_updates.display()

####Data Quality checks

In [0]:
row_count = dim_automaker_updates.count()

In [0]:
null_key_count = dim_automaker_updates.filter(col("Automaker_ID").isNull()).count()

In [0]:
duplicate_key_count = dim_automaker_updates.groupBy("Automaker_ID").count().filter("count > 1").count()

In [0]:
print(f"row count: {row_count}")
print(f"null Automaker_ID count: {null_key_count}")
print(f"duplicate Automaker_ID count: {duplicate_key_count}")

In [0]:
assert null_key_count == 0, "Automaker_ID should never be null in dim_automaker"
assert duplicate_key_count == 0, "Automaker_ID should be unique in dim_automaker"

In [0]:
if DeltaTable.isDeltaTable(spark, GOLD_TABLE_PATH):
 
    dim_automaker_table = DeltaTable.forPath(spark, GOLD_TABLE_PATH)
 
    (dim_automaker_table.alias("t")
        .merge(dim_automaker_updates.alias("s"), "t.Automaker_ID = s.Automaker_ID")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
 
else:
 
    dim_automaker_updates.write \
        .format("delta") \
        .mode("overwrite") \
        .save(GOLD_TABLE_PATH)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE_NAME}
    USING DELTA
    LOCATION '{GOLD_TABLE_PATH}'
""")

In [0]:
spark.sql(f"OPTIMIZE {GOLD_TABLE_NAME}")